In [ ]:
import numpy as np
from typing import Iterable, List, Tuple, Union
from netCDF4 import Dataset
import datetime

try:
    from pyproj import CRS, Transformer
    _HAS_PYPROJ = True
except Exception:
    _HAS_PYPROJ = False

In [ ]:
def _coriolis_f_at_75deg() -> float:
    """
    Параметр Кориолиса f на широте 75° (с^-1).
    """
    omega = 7.2921159e-5  # скор. вращения Земли, с^-1
    phi = np.deg2rad(75.0)
    return 2.0 * omega * np.sin(phi)

In [ ]:
def _drag_coefficient_large_pond(U10: float) -> float:
    """
    Коэффициент трения воздуха C_d по Large & Pond (1981) для 10-метрового ветра.
    Возвращает безразмерную величину.
    """
    U = np.maximum(U10, 0.0)
    if U < 3.0:
        return 0.0010
    elif U <= 20.0:
        return (0.63 + 0.066 * U) * 1e-3
    else:
        return 0.0026

In [ ]:
def _wind_stress(u10: float, v10: float, rho_air: float = 1.225) -> Tuple[float, float]:
    """
    Ветровой стресс τ = ρ_air * C_d * |U| * (u, v), Н/м^2.
    Возвращает (tau_x, tau_y).
    """
    U = np.hypot(u10, v10)
    Cd = _drag_coefficient_large_pond(U)
    factor = rho_air * Cd * U
    return factor * u10, factor * v10

In [ ]:
def ekman_displacement_km(
    lat_deg: float,
    lon_deg: float,
    u_wind_ms: float,
    v_wind_ms: float,
    dt_seconds: float,
    H_m: float = 20.0,
    rho_w: float = 1025.0,
    use_pyproj: bool = True,
) -> Tuple[float, float, float, float]:
    """
    Рассчитывает экмановское смещение точки за интервал времени dt_seconds.
    Выход: (dx_km, dy_km, lat_new, lon_new)
    """
    if dt_seconds <= 0:
        raise ValueError("dt_seconds должен быть > 0")

    # 1) Кориолис на 75°
    f = _coriolis_f_at_75deg()

    # 2) Ветровой стресс
    tau_x, tau_y = _wind_stress(u_wind_ms, v_wind_ms)

    # 3) Экмановская скорость (усреднённая по H): U_E = (1/(rho_w f H)) * (τ × k)
    coeff = 1.0 / (rho_w * f * H_m)
    Ue_x = coeff * (tau_y)   # м/с
    Ue_y = coeff * (-tau_x)  # м/с

    # 4) Смещение за dt
    dx_m = Ue_x * dt_seconds
    dy_m = Ue_y * dt_seconds
    dx_km = dx_m / 1000.0
    dy_km = dy_m / 1000.0

    # 5) Новые координаты
    if use_pyproj and _HAS_PYPROJ:
        crs_geodetic = CRS.from_epsg(4326)
        crs_local = CRS.from_proj4(f"+proj=aeqd +lat_0={lat_deg} +lon_0={lon_deg} +datum=WGS84 +units=m +no_defs")
        to_local = Transformer.from_crs(crs_geodetic, crs_local, always_xy=True)
        to_geo = Transformer.from_crs(crs_local, crs_geodetic, always_xy=True)

        x0, y0 = to_local.transform(lon_deg, lat_deg)
        x1, y1 = x0 + dx_m, y0 + dy_m
        lon_new, lat_new = to_geo.transform(x1, y1)
        lon_new = (lon_new + 180.0) % 360.0 - 180.0
    else:
        R_E = 6371000.0
        dlat = (dy_m / R_E) * (180.0 / np.pi)
        dlon = (dx_m / (R_E * np.cos(np.deg2rad(lat_deg)))) * (180.0 / np.pi)
        lat_new = lat_deg + dlat
        lon_new = (lon_deg + dlon + 180.0) % 360.0 - 180.0

    return float(dx_km), float(dy_km), float(lat_new), float(lon_new)

In [ ]:
# Пример использования:
lat0, lon0 = 75.0, 30.0       # градусы
u10, v10 = 8.0, 4.0           # м/с
dt = 6 * 3600.0               # 6 часов

dx_km, dy_km, lat2, lon2 = ekman_displacement_km(lat0, lon0, u10, v10, dt)
print(f"Смещение (км): dx={dx_km:.3f}, dy={dy_km:.3f}")
print(f"Новые координаты: lat={lat2:.6f}, lon={lon2:.6f}")

In [ ]:
def get_wind_components_at_points(
    lat_grid: np.ndarray,
    lon_grid: np.ndarray,
    u: np.ndarray,
    v: np.ndarray,
    points: Iterable[Tuple[float, float]],
    wrap_longitude: bool = True,
    outside: str = "nan",
) -> np.ndarray:
    """
    Возвращает компоненты скорости ветра (u, v) для набора точек (широта, долгота),
    интерполируя поле u и v на регулярной сетке (meshgrid) билинейно.

    Параметры:
      - lat_grid: 2D массив широт (как из np.meshgrid), совпадает по форме с lon_grid, u, v.
      - lon_grid: 2D массив долгот (как из np.meshgrid), совпадает по форме с lat_grid, u, v.
      - u: 2D массив компоненты ветра по долготе (ось x), та же форма.
      - v: 2D массив компоненты ветра по широте (ось y), та же форма.
      - points: Iterable из кортежей (широта, долгота) — порядок обязательный: (lat, lon).
      - wrap_longitude: если True — долготы входных точек приводятся к диапазону сетки
                        (0..360 или -180..180).
      - outside: "nan" — вернуть NaN для точки вне диапазона; "clamp" — зажать к границе;
                 "raise" — вызвать исключение.

    Возвращает:
      - np.ndarray формы (N, 2), где N — число точек; столбцы: [u_interp, v_interp].

    Примечания:
      - Используется билинейная интерполяция по четырем соседним узлам.
      - Предполагается регулярная монотонная сетка по широте и долготе.
    """
    # Проверка формы
    if lat_grid.shape != lon_grid.shape or lat_grid.shape != u.shape or lat_grid.shape != v.shape:
        raise ValueError("lat_grid, lon_grid, u и v должны иметь одинаковую 2D-форму.")

    # Оси широты/долготы из meshgrid (строки — широты, столбцы — долготы)
    lat_axis = lat_grid[:, 0]
    lon_axis = lon_grid[0, :]

    def axis_bounds(axis: np.ndarray):
        asc = axis[0] < axis[-1]
        return (min(axis[0], axis[-1]), max(axis[0], axis[-1]), asc)

    lat_min, lat_max, lat_asc = axis_bounds(lat_axis)
    lon_min, lon_max, lon_asc = axis_bounds(lon_axis)

    def normalize_lon(lon: float) -> float:
        if not wrap_longitude:
            return lon
        # Типичная глобальная сетка: либо 0..360, либо -180..180
        if lon_min >= 0 and lon_max <= 360:
            return lon % 360.0
        else:
            return ((lon + 180.0) % 360.0) - 180.0

    def is_outside(x: float, xmin: float, xmax: float) -> bool:
        return (x < xmin) or (x > xmax)

    def bracket_and_weight(axis: np.ndarray, x: float, asc: bool):
        """
        Возвращает индексы (i0, i1) и вес w в [0,1], такие что
        x = axis[i0] + w * (axis[i1] - axis[i0]).
        """
        n = axis.size
        if n < 2:
            raise ValueError("Длина оси должна быть >= 2.")
        if asc:
            i = np.searchsorted(axis, x)
            if i == 0:
                i0, i1 = 0, 1
            elif i >= n:
                i0, i1 = n - 2, n - 1
            else:
                i0, i1 = i - 1, i
        else:
            axis_rev = axis[::-1]
            i = np.searchsorted(axis_rev, x)
            if i == 0:
                i0_rev, i1_rev = 0, 1
            elif i >= n:
                i0_rev, i1_rev = n - 2, n - 1
            else:
                i0_rev, i1_rev = i - 1, i
            i0 = n - 1 - i1_rev
            i1 = n - 1 - i0_rev

        x0, x1 = axis[i0], axis[i1]
        w = 0.0 if x1 == x0 else (x - x0) / (x1 - x0)
        return i0, i1, float(w)

    # Преобразуем точки в список один раз
    pts: List[Tuple[float, float]] = list(points)
    N = len(pts)
    comps = np.empty((N, 2), dtype=float)

    for k, (lat_p, lon_p) in enumerate(pts):
        lon_p = normalize_lon(float(lon_p))
        lat_p = float(lat_p)

        out_lat = is_outside(lat_p, lat_min, lat_max)
        out_lon = is_outside(lon_p, lon_min, lon_max)

        if (out_lat or out_lon):
            if outside == "nan":
                comps[k, :] = np.nan
                continue
            elif outside == "raise":
                raise ValueError(f"Точка вне диапазона сетки: lat={lat_p}, lon={lon_p}")
            elif outside == "clamp":
                lat_p = min(max(lat_p, lat_min), lat_max)
                lon_p = min(max(lon_p, lon_min), lon_max)
            else:
                raise ValueError("Параметр 'outside' должен быть 'nan', 'clamp' или 'raise'.")

        i0_lat, i1_lat, t = bracket_and_weight(lat_axis, lat_p, lat_asc)
        i0_lon, i1_lon, s = bracket_and_weight(lon_axis, lon_p, lon_asc)

        # Билинейная интерполяция
        u00 = u[i0_lat, i0_lon]; u01 = u[i0_lat, i1_lon]
        u10 = u[i1_lat, i0_lon]; u11 = u[i1_lat, i1_lon]
        v00 = v[i0_lat, i0_lon]; v01 = v[i0_lat, i1_lon]
        v10 = v[i1_lat, i0_lon]; v11 = v[i1_lat, i1_lon]

        u_interp = (1 - s) * (1 - t) * u00 + s * (1 - t) * u01 + (1 - s) * t * u10 + s * t * u11
        v_interp = (1 - s) * (1 - t) * v00 + s * (1 - t) * v01 + (1 - s) * t * v10 + s * t * v11

        comps[k, 0] = float(u_interp)
        comps[k, 1] = float(v_interp)

    return comps

In [ ]:
lon_min_era5, lon_max_era5, lat_min_era5, lat_max_era5 = 35, 105, 66, 82
lat1, lat2 = (90-lat_max_era5)*4, (90-lat_min_era5)*4+1
lon1, lon2 = lon_min_era5*4, lon_max_era5*4+1

In [ ]:
month = '2024-09'
file = f'/mnt/hippocamp/DATA/ERA5/w10/era5_uv10m_{month}.nc'
data = Dataset(file, 'r')

tt = np.asarray(data.variables['valid_time'])
time = np.asarray([datetime.datetime(1970, 1, 1, 0, 0, 0) + datetime.timedelta(seconds=int(t)) for t in tt])
u10 = data.variables['u10'][:,lat1:lat2,lon1:lon2]
v10 = data.variables['v10'][:,lat1:lat2,lon1:lon2]

longitude = np.array(data.variables['longitude'][lon1:lon2])
latitude = np.array(data.variables['latitude'][lat1:lat2])
lon_grid, lat_grid = np.meshgrid(longitude, latitude)

data.close()

In [ ]:
t = 168
u = u10[t]
v = v10[t]

points = [
        (76, 70),     # форма: (широта, долгота)
    ]

uv = get_wind_components_at_points(lat_grid, lon_grid, u, v, points, wrap_longitude=True, outside="clamp")
print(uv)  # форма: (N, 2), столбцы [u, v]

In [ ]:
latitude[(lat_max_era5-76)*4], longitude[(70-lon_min_era5)*4]

In [ ]:
u10[t, (lat_max_era5-76)*4, (70-lon_min_era5)*4], v10[t, (lat_max_era5-76)*4, (70-lon_min_era5)*4]

In [ ]:
date_start = datetime.datetime(2024, 9, 6, 12, 0)
date_finish = datetime.datetime(2024, 9, 10, 12, 0)
points = [
    (76.375, 72),
    (76.375, 74),
    (76.375, 76),
    (76.375, 78),
    (76.375, 80),
]
t_start = np.where(time == date_start)[0][0]
t_finish = np.where(time == date_finish)[0][0]

In [ ]:
points_coords = []
for t in range(t_start, t_finish):
